<a href="https://colab.research.google.com/github/Abhyuday19/Pyspark_All_Practice_Sessions/blob/main/Pysparka_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
## Installing Pyspark in google collab
!pip install Pyspark
## importing required libraries
from pyspark.sql import SparkSession
## Create spark session
spark = SparkSession.builder \
        .appName("RetailPro Analytics") \
        .master("local[*]") \
        .getOrCreate()

## Way to verify the spark is working
print(f"Spark Version : {spark.version}")
print(f"Spark Session created sucessfuly!")

Spark Version : 4.0.5
Spark Session created sucessfuly!


In [11]:
## loading the excel file usoing the dataframe.
orders_df = spark.read.csv(
    "ecommerce_orders.csv",
    header= True,
    inferSchema = True
)
# Displaying the first row
## header=True means that the first row of your CSV file contains column names, not actual data
orders_df.show()
## Show the Schema
orders_df.printSchema()

## Count the total records
print(f"Total orders: {orders_df.count()}")

+--------+----------+-----------+------------+-----------+--------+----------+----------------+------------+---------+---------+--------------+-------------+
|order_id|order_date|customer_id|product_name|   category|quantity|unit_price|discount_percent|total_amount|  country|   status|payment_method|shipping_days|
+--------+----------+-----------+------------+-----------+--------+----------+----------------+------------+---------+---------+--------------+-------------+
|    1001|2025-01-15|       5023|      LAPTOP|Electronics|       2|    899.99|              10|     1619.98|      USA|Completed|   Credit Card|            5|
|    1002|2025-01-18|       5012|  HEADPHONES|Electronics|       1|     79.99|               0|       79.99|       UK|Completed|        PayPal|            3|
|    1003|2025-02-03|       5045|    KEYBOARD|Electronics|       3|     49.99|              15|      127.47|   Canada|  Shipped|    Debit Card|            7|
|    1004|2025-02-10|       5008|     MONITOR|Electr

In [12]:
## Checking how much unique customers the data have
unique_customers = orders_df.select("customer_id").distinct().count()
print(f"Total_number_of_Unique_Customers: {unique_customers}")

Total_number_of_Unique_Customers: 46


In [15]:
## To check the data range , when the order occurs
## Min_date it means the first order , and the Max_date means that the lastest or the current latest order.

from pyspark.sql.functions import min , max

date_range = orders_df.select(min("order_date").alias("earliest_orders"),
                              max("order_date").alias("latest_orders"))
date_range.show()

+---------------+-------------+
|earliest_orders|latest_orders|
+---------------+-------------+
|     2025-01-15|   2025-11-30|
+---------------+-------------+



In [17]:
## To have the idea of rthe distribution across the different categories
## Countig the orders by the Categories wise as per data.
orders_df.groupBy("category").count().orderBy("count",asceding=False).show()

+-----------+-----+
|   category|count|
+-----------+-----+
| Stationery|   16|
|Accessories|   16|
|  Furniture|   24|
|Electronics|   44|
+-----------+-----+



In [20]:
## To have the idea of the bussines operating in which countries
## Counting the orders by the countries name.
orders_df.groupBy("country").count().orderBy("count",ascending=False).show()

+---------+-----+
|  country|count|
+---------+-----+
|  Germany|   13|
|      USA|   13|
|       UK|   13|
|   Canada|   13|
|   France|   12|
|    India|   12|
|    Japan|   12|
|Australia|   12|
+---------+-----+



In [24]:
## Checking the data quality issues in the data (like the missing values , the null values)
## col, sum, and when are functions used to work with DataFrame columns. They help you select columns, perform calculations, and apply conditions.
## the sum function is changed to _sum , as because to avoid missunderstanding of the sum function
from pyspark.sql.functions import col , sum as _sum , when
## Checking for the null values in the column
orders_df.select([
    _sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in orders_df.columns
]).show()

+--------+----------+-----------+------------+--------+--------+----------+----------------+------------+-------+------+--------------+-------------+
|order_id|order_date|customer_id|product_name|category|quantity|unit_price|discount_percent|total_amount|country|status|payment_method|shipping_days|
+--------+----------+-----------+------------+--------+--------+----------+----------------+------------+-------+------+--------------+-------------+
|       0|         0|          0|           0|       0|       0|         0|               0|           0|      0|     0|             0|           19|
+--------+----------+-----------+------------+--------+--------+----------+----------------+------------+-------+------+--------------+-------------+



In [26]:
## printing the Schema of the data set
orders_df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- country: string (nullable = true)
 |-- status: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- shipping_days: integer (nullable = true)



In [27]:
## From now answerign the busssiness questions using spark
## total orders
total_orders = orders_df.count()
print(f"Total Orders: {total_orders}")

Total Orders: 100


In [32]:
## Now we are having the quick summary of the data set
## it helps us to understand the count , mean , standard deviation , min values , max values , helps to understand the range and the distribution of values like orders amount , prices , quantities.
orders_df.describe().show()

+-------+------------------+------------------+------------+-----------+-----------------+------------------+-----------------+------------------+---------+---------+--------------+------------------+
|summary|          order_id|       customer_id|product_name|   category|         quantity|        unit_price| discount_percent|      total_amount|  country|   status|payment_method|     shipping_days|
+-------+------------------+------------------+------------+-----------+-----------------+------------------+-----------------+------------------+---------+---------+--------------+------------------+
|  count|               100|               100|         100|        100|              100|               100|              100|               100|      100|      100|           100|                81|
|   mean|            1050.5|           5026.84|        NULL|       NULL|             2.56|177.12999999999988|              9.2|236.62360000000004|     NULL|     NULL|          NULL|  6.54320987654

In [33]:
## Now we are showing the data having the high value ordrs (considering the values more than the 500 dollars)
## this filter transformation is being used to create the new data frame having all those rows that match the required condition.
## It waits till we call the show action before implementing anything on the code.
from pyspark.sql.functions import col
high_values = orders_df.filter(col("total_amount")> 500)
high_values.show()

+--------+----------+-----------+------------+-----------+--------+----------+----------------+------------+-------+---------+--------------+-------------+
|order_id|order_date|customer_id|product_name|   category|quantity|unit_price|discount_percent|total_amount|country|   status|payment_method|shipping_days|
+--------+----------+-----------+------------+-----------+--------+----------+----------------+------------+-------+---------+--------------+-------------+
|    1001|2025-01-15|       5023|      LAPTOP|Electronics|       2|    899.99|              10|     1619.98|    USA|Completed|   Credit Card|            5|
|    1007|2025-03-08|       5041|        DESK|  Furniture|       2|    349.99|               0|      699.98|  Japan|Completed|   Credit Card|            8|
|    1013|2025-04-20|       5011|      LAPTOP|Electronics|       1|    899.99|              20|      719.99| France|Completed|        PayPal|            3|
|    1025|2025-07-12|       5018|      LAPTOP|Electronics|      

In [36]:
## Considering above condition and then applying the filter in order to apply the more restrictive condition.
## Considering the two condtions only now the total_amount to be in between 500 to 1000 and the category to be electronics.
electronics_high_values = orders_df.filter(
    (col("total_amount") > 500) &
    (col("total_amount")<1000) &
    (col("category")== "Electronics")
)
electronics_high_values.show()

+--------+----------+-----------+------------+-----------+--------+----------+----------------+------------+-------+---------+--------------+-------------+
|order_id|order_date|customer_id|product_name|   category|quantity|unit_price|discount_percent|total_amount|country|   status|payment_method|shipping_days|
+--------+----------+-----------+------------+-----------+--------+----------+----------------+------------+-------+---------+--------------+-------------+
|    1013|2025-04-20|       5011|      LAPTOP|Electronics|       1|    899.99|              20|      719.99| France|Completed|        PayPal|            3|
|    1025|2025-07-12|       5018|      LAPTOP|Electronics|       1|    899.99|               0|      899.99|    USA|Completed|   Credit Card|            2|
|    1049|2025-03-01|       5019|      LAPTOP|Electronics|       1|    899.99|              10|      809.99|    USA|Completed|   Credit Card|            3|
|    1061|2025-08-06|       5042|      LAPTOP|Electronics|      

In [37]:
## Now more precisive in the conditions
sales_report = orders_df.select("order_date", "product_name", "total_amount")
sales_report.show(10)

+----------+------------+------------+
|order_date|product_name|total_amount|
+----------+------------+------------+
|2025-01-15|      LAPTOP|     1619.98|
|2025-01-18|  HEADPHONES|       79.99|
|2025-02-03|    KEYBOARD|      127.47|
|2025-02-10|     MONITOR|      239.99|
|2025-02-14|       MOUSE|      142.45|
|2025-03-02|       CHAIR|      179.99|
|2025-03-08|        DESK|      699.98|
|2025-03-15|        LAMP|      113.97|
|2025-03-22|    NOTEBOOK|        49.9|
|2025-03-25|     PEN_SET|       46.76|
+----------+------------+------------+
only showing top 10 rows


In [39]:
## Alternative Approach to change the column name
orders_renamed = orders_df.withColumnRenamed("total_column", "revenue")
orders_renamed.show(5)

+--------+----------+-----------+------------+-----------+--------+----------+----------------+------------+-------+---------+--------------+-------------+
|order_id|order_date|customer_id|product_name|   category|quantity|unit_price|discount_percent|total_amount|country|   status|payment_method|shipping_days|
+--------+----------+-----------+------------+-----------+--------+----------+----------------+------------+-------+---------+--------------+-------------+
|    1001|2025-01-15|       5023|      LAPTOP|Electronics|       2|    899.99|              10|     1619.98|    USA|Completed|   Credit Card|            5|
|    1002|2025-01-18|       5012|  HEADPHONES|Electronics|       1|     79.99|               0|       79.99|     UK|Completed|        PayPal|            3|
|    1003|2025-02-03|       5045|    KEYBOARD|Electronics|       3|     49.99|              15|      127.47| Canada|  Shipped|    Debit Card|            7|
|    1004|2025-02-10|       5008|     MONITOR|Electronics|      